# KL vs VQ First-Stage Autoencoder Comparison
## Experiment 1: No_Mixed Dataset

**FIXED VERSION** - Based on documented issues in `docs/KAGGLE_ISSUES_LOG.md`

This notebook trains both KL and VQ autoencoders and compares their reconstruction quality.

**Kaggle Setup:**
1. Enable GPU T4 (single GPU)
2. Add the No_Mixed dataset to the notebook
3. Clone/pull the Mixed-Pixels repository
4. Run all cells

**Expected Runtime:** ~2-3 hours (50 epochs each)

## 1. Environment Setup

In [ ]:
import os
import sys
from pathlib import Path

# Kaggle paths
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_OUTPUT = Path('/kaggle/output')

# Check if running on Kaggle
IS_KAGGLE = KAGGLE_INPUT.exists()

print(f"Running on Kaggle: {IS_KAGGLE}")
print(f"Working directory: {os.getcwd()}")

if IS_KAGGLE:
    print(f"\nAvailable datasets in /kaggle/input:")
    for item in KAGGLE_INPUT.iterdir():
        print(f"  - {item.name}")

In [ ]:
# Clone or navigate to repository
if IS_KAGGLE:
    REPO_NAME = 'Mixed_Pixel'  # Adjust to your repo name
    
    if not (KAGGLE_WORKING / REPO_NAME).exists():
        print("Cloning repository...")
        # Clone from GitHub - UPDATE THIS URL!
        !git clone https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git /kaggle/working/Mixed_Pixel
        
    os.chdir(KAGGLE_WORKING / REPO_NAME)
    print(f"Changed to: {os.getcwd()}")
    
    # Add src to path
    sys.path.insert(0, str(Path.cwd() / 'src'))
else:
    # Local development
    repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    sys.path.insert(0, str(repo_root / 'src'))

print(f"\nPython path includes: {sys.path[0]}")

In [ ]:
# Check GPU availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
        props = torch.cuda.get_device_properties(i)
        print(f"  Total memory: {props.total_memory / 1e9:.2f} GB")
        print(f"  Compute capability: {props.major}.{props.minor}")
    
    print(f"\n✓ Will use single GPU for training (DataParallel issues with KL model)")

## 2. Configure Paths for Kaggle

**CRITICAL FIXES APPLIED:**
- ✅ Dataset path from Issue #2
- ✅ Persistent storage from Issue #9
- ✅ Speed optimization from Issue #10
- ✅ Memory management from Issue #8

In [ ]:
import yaml

# Load base configurations
config_dir = Path('configs/experiment1')
kl_config_path = config_dir / 'kl_autoencoder.yaml'
vq_config_path = config_dir / 'vq_autoencoder.yaml'

with open(kl_config_path, 'r') as f:
    kl_config = yaml.safe_load(f)

with open(vq_config_path, 'r') as f:
    vq_config = yaml.safe_load(f)

# Override paths for Kaggle
if IS_KAGGLE:
    # FIX #1: VERIFIED dataset path from Issue #2
    # Path structure: /kaggle/input/datasets/tejabulusu/no-mixed/No_mixed/Image1.jpg
    dataset_root = KAGGLE_INPUT / 'datasets' / 'tejabulusu' / 'no-mixed' / 'No_mixed'
    
    # Verify path exists and has images
    if not dataset_root.exists():
        print(f"❌ ERROR: Dataset not found at {dataset_root}")
        print(f"\nAvailable in /kaggle/input/:")
        for item in KAGGLE_INPUT.rglob('*'):
            if item.is_dir():
                print(f"  [DIR]  {item}")
        raise FileNotFoundError(f"Dataset not found at {dataset_root}")
    
    print(f"✓ Dataset found at: {dataset_root}")
    
    # Count images to verify
    jpg_count = len(list(dataset_root.glob('*.jpg')))
    png_count = len(list(dataset_root.glob('*.png')))
    total_images = jpg_count + png_count
    
    print(f"  Found {jpg_count} JPG and {png_count} PNG images (Total: {total_images})")
    
    if total_images == 0:
        raise ValueError("No images found in dataset directory!")
    
    # Update both configs with ALL fixes
    for config in [kl_config, vq_config]:
        # Store dataset path (will be restructured in next cell)
        config['data']['dataset_root'] = str(dataset_root)
        
        # FIX #2: Use /kaggle/output for PERSISTENT storage (Issue #9)
        exp_name = config['experiment']['name']
        config['paths']['checkpoint_dir'] = f'/kaggle/output/checkpoints/{exp_name}'
        config['paths']['log_dir'] = f'/kaggle/output/logs/{exp_name}'
        config['paths']['tensorboard_dir'] = f'/kaggle/output/runs/{exp_name}'
        config['paths']['output_dir'] = f'/kaggle/output/outputs/{exp_name}'
        
        # FIX #3: Optimize data loading for Kaggle (Issue #10)
        config['data']['num_workers'] = 0  # Single-threaded (10x faster!)
        config['data']['pin_memory'] = False  # Not needed for single GPU
        
        # FIX #4: Memory management (Issue #8)
        config['data']['batch_size'] = 2  # Reduced for single GPU
        
        # FIX #5: Reduce epochs to fit in 12-hour session
        config['training']['max_epochs'] = 50  # Instead of 100
        
        # Create output directories
        for path_key in ['checkpoint_dir', 'log_dir', 'tensorboard_dir', 'output_dir']:
            Path(config['paths'][path_key]).mkdir(parents=True, exist_ok=True)

print("\n" + "="*70)
print("✓ Configurations updated for Kaggle")
print("="*70)
print(f"  Dataset source: {kl_config['data']['dataset_root']}")
print(f"  Images found: {total_images}")
print(f"  Checkpoints: {kl_config['paths']['checkpoint_dir']} (PERSISTENT ✓)")
print(f"  Batch size: {kl_config['data']['batch_size']}")
print(f"  Num workers: {kl_config['data']['num_workers']}")
print(f"  Max epochs: {kl_config['training']['max_epochs']}")
print(f"\n⚠️  IMPORTANT: Checkpoints in /kaggle/output/ are downloadable after session ends!")
print("="*70)

## 3. Restructure Dataset

**Fix for Issue #3:** ThermalImageDataset expects class folders, but Kaggle data is flat

In [ ]:
from data import ThermalImageDataset
import shutil

# Create temporary structure with dummy class folder
temp_dataset_path = KAGGLE_WORKING / 'dataset_structured'
temp_dataset_path.mkdir(exist_ok=True)

class_folder = temp_dataset_path / 'no_mixed'

if not class_folder.exists():
    class_folder.mkdir()
    
    source_path = Path(kl_config['data']['dataset_root'])
    
    print(f"Restructuring dataset for PyTorch...")
    print(f"  Source: {source_path}")
    print(f"  Target: {class_folder}")
    
    # Create symlinks (faster than copying)
    image_count = 0
    for ext in ['*.jpg', '*.png', '*.jpeg']:
        for img in source_path.glob(ext):
            link_path = class_folder / img.name
            if not link_path.exists():
                link_path.symlink_to(img)
                image_count += 1
    
    print(f"✓ Created structure with {image_count} images")
else:
    image_count = len(list(class_folder.iterdir()))
    print(f"✓ Structure already exists with {image_count} images")

# Update configs to use restructured path
for config in [kl_config, vq_config]:
    config['data']['dataset_root'] = str(temp_dataset_path)

print(f"\n✓ New dataset root: {kl_config['data']['dataset_root']}")

# Verify dataset loads correctly
test_dataset = ThermalImageDataset(
    root_dir=temp_dataset_path,
    transform=None,
    load_mode='rgb'
)
print(f"✓ Dataset verification: {len(test_dataset)} images loaded successfully")

if len(test_dataset) == 0:
    raise ValueError("Dataset loading failed - 0 images found!")

## 4. Import Training Modules

**Fix for Issue #1:** Import from correct API module

In [ ]:
# Import training API (supports automatic GPU detection)
try:
    from scripts.training.train_first_stage_api import train_first_stage
    from models.ldm.first_stage.kl.model import KLAutoencoder
    from models.ldm.first_stage.vq.model import VQAutoencoder
    
    print("✓ Successfully imported training modules")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("\nTroubleshooting:")
    print(f"  Current directory: {os.getcwd()}")
    print(f"  Python path: {sys.path[0]}")
    
    src_path = Path('src')
    if src_path.exists():
        print("\n  Available modules:")
        for item in ['models/ldm/first_stage', 'scripts/training']:
            full_path = src_path / item
            if full_path.exists():
                print(f"    ✓ {item}")
                for f in full_path.glob('*.py'):
                    print(f"      - {f.name}")
    raise

## 5. Clear GPU Memory

**Fix for Issue #5:** Clear memory before training

In [ ]:
import gc

# Delete any existing models
for var in ['model', 'kl_model', 'vq_model']:
    if var in locals():
        del locals()[var]

# Force garbage collection
gc.collect()

# Clear CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✓ GPU memory cleared")
    print(f"  Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"  Cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

## 6. Train KL Autoencoder

**Fixes Applied:**
- Issue #4: Force single GPU (DataParallel incompatible)
- Issue #8: Reduced batch size
- Issue #9: Save to /kaggle/output/
- Issue #10: Optimized data loading

In [ ]:
# Force single GPU to avoid DataParallel issues
torch.cuda.device_count = lambda: 1

print("="*70)
print("Training KL-Regularized Autoencoder (Single GPU, Optimized)")
print("="*70)

# Ensure single GPU
kl_config['hardware']['device'] = 'cuda:0'

# Train KL model
kl_results = train_first_stage(
    config=kl_config,
    model_type='kl'
)

print("\n" + "="*70)
print("✓ KL Training Complete")
print("="*70)
print(f"  Final validation loss: {kl_results.get('final_val_loss', 'N/A'):.6f}")
print(f"  Best epoch: {kl_results.get('best_epoch', 'N/A')}")
print(f"  Checkpoints: {kl_config['paths']['checkpoint_dir']}")
print(f"\n✓ Checkpoints saved to /kaggle/output/ - downloadable after session!")
print("="*70)

## 7. Train VQ Autoencoder

In [ ]:
# Clear GPU memory before VQ training
if 'model' in locals():
    del model
gc.collect()
torch.cuda.empty_cache()

print(f"GPU memory after cleanup: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB\n")

# Force single GPU for consistency
torch.cuda.device_count = lambda: 1

print("="*70)
print("Training VQ-Regularized Autoencoder (Single GPU, Optimized)")
print("="*70)

vq_config['hardware']['device'] = 'cuda:0'

# Train VQ model
vq_results = train_first_stage(
    config=vq_config,
    model_type='vq'
)

print("\n" + "="*70)
print("✓ VQ Training Complete")
print("="*70)
print(f"  Final validation loss: {vq_results.get('final_val_loss', 'N/A'):.6f}")
print(f"  Best epoch: {vq_results.get('best_epoch', 'N/A')}")
print(f"  Checkpoints: {vq_config['paths']['checkpoint_dir']}")
print(f"\n✓ Checkpoints saved to /kaggle/output/ - downloadable after session!")
print("="*70)

## 8. Experiment Summary

In [ ]:
print("="*70)
print("EXPERIMENT COMPLETE - Summary")
print("="*70)

print("\n✓ Both models trained successfully!")

print("\n📊 Training Results:")
print(f"  KL - Best val loss: {kl_results.get('final_val_loss', 'N/A'):.6f} (epoch {kl_results.get('best_epoch', 'N/A')})")
print(f"  VQ - Best val loss: {vq_results.get('final_val_loss', 'N/A'):.6f} (epoch {vq_results.get('best_epoch', 'N/A')})")

print("\n📁 Output Locations (in /kaggle/output/):")
print(f"  KL checkpoints: {kl_config['paths']['checkpoint_dir']}")
print(f"  VQ checkpoints: {vq_config['paths']['checkpoint_dir']}")
print(f"  KL logs: {kl_config['paths']['log_dir']}")
print(f"  VQ logs: {vq_config['paths']['log_dir']}")

print("\n📥 Next Steps:")
print("  1. Click 'Output' tab in Kaggle")
print("  2. Download checkpoints/ and logs/ folders")
print("  3. Run local evaluation with full metrics")
print("  4. Compare KL vs VQ boundary preservation")
print("  5. Update docs/CURRENT_STATE.md with findings")
print("  6. Document decision in docs/DECISIONS.md")

print("\n🎯 Analysis Questions:")
print("  - Which model has lower reconstruction error?")
print("  - Which preserves thermal boundaries better?")
print("  - Is VQ codebook utilization acceptable?")
print("  - Which should be used for LDM diffusion stage?")

print("\n" + "="*70)
print("✅ All training complete - Ready for evaluation!")
print("="*70)